In [48]:
import torch
import torch.nn as nn
import torch.optim as optim

In [49]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cuda


In [50]:
with open("data/input.txt", "r") as f:
    text = f.read()

print(len(text))
print(text[:500])

1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor


In [51]:
chars = sorted(list(set(text)))

vocab_size = len(chars)

print("Vocab size: ", vocab_size)
print(chars)

Vocab size:  65
['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [52]:
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i : ch for i, ch in enumerate(chars)}

In [53]:
data = torch.tensor([stoi[ch] for ch in text], dtype=torch.long)

print(data.shape)
print(data[:20])

torch.Size([1115394])
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56])


In [54]:
batch_size = 64
block_size = 64

x = data[:block_size]
y = data[1:block_size + 1]

print(x.shape)
print(y.shape)

torch.Size([64])
torch.Size([64])


In [55]:
def get_batch():
    ix = torch.randint(len(data) - block_size, (batch_size,))

    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])

    return x, y

In [56]:
x, y = get_batch()

print(x.shape)
print(y.shape)
print(x[0].shape)

torch.Size([64, 64])
torch.Size([64, 64])
torch.Size([64])


In [57]:
print("".join(itos[int(i)] for i in x[0]))
print("".join(itos[int(i)] for i in y[0]))

ow the cause why music was ordain'd!
Was it not to refresh the m
w the cause why music was ordain'd!
Was it not to refresh the mi


In [58]:


class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        self.lstm = nn.LSTM(embedding_dim, hidden_size, batch_first=True)

        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        x = self.embedding(x)

        x,(h, c) = self.lstm(x)

        x = self.fc(x)

        return x

In [59]:
embedding_dim = 128
hidden_size = 256

model = LSTMModel(vocab_size, embedding_dim, hidden_size).to(device)

In [60]:
print(next(model.parameters()).device)

cuda:0


In [61]:
x, y = get_batch()
x = x.to(device)
y = y.to(device)

output = model(x)

print(x.shape)
print(output.shape)

torch.Size([64, 64])
torch.Size([64, 64, 65])


In [62]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    model.parameters(),
    lr=0.001
)

In [63]:
epochs = 10

for epoch in range(epochs):
    model.train()

    total_loss = 0

    for step in range(1000):

        x, y = get_batch()

        x = x.to(device)
        y = y.to(device)
        
        optimizer.zero_grad()
        
        output = model(x)
        
        loss = criterion(output.view(-1, vocab_size), y.view(-1))
        
        loss.backward()
        
        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch+1}: "
        f"Loss {total_loss / 1000:.4f}"
        f"Device {device}: "
    )

Epoch 1: Loss 1.8494Device cuda: 
Epoch 2: Loss 1.4924Device cuda: 
Epoch 3: Loss 1.4107Device cuda: 
Epoch 4: Loss 1.3679Device cuda: 
Epoch 5: Loss 1.3372Device cuda: 
Epoch 6: Loss 1.3143Device cuda: 
Epoch 7: Loss 1.2978Device cuda: 
Epoch 8: Loss 1.2831Device cuda: 
Epoch 9: Loss 1.2681Device cuda: 
Epoch 10: Loss 1.2575Device cuda: 


In [70]:
@torch.no_grad()
def generate(model, prompt, max_new_tokens):

    model.eval()

    idx = torch.tensor([stoi[ch] for ch in prompt], dtype=torch.long).unsqueeze(0).to(device)

    for _ in range(max_new_tokens):

        logits = model(idx)
        logits = logits[:, -1, :]

        probs = torch.softmax(logits, dim=-1)

        next_token = torch.multinomial(probs, num_samples=1)

        idx = torch.cat((idx, next_token), dim=1)

    result = "".join(itos[int(i)] for i in idx[0])

    return result

In [71]:
print(generate(model, "ROMEO:", 500))

ROMEO:
'Tell me from me now place, and willing to her!
Ay, do at my names night there,
With he will atts done: I must bit your daughter forget
the fairest stones, come, we am concived
The ghost wilt rise, and myself; I have, to know
That learn'd as I was; in a man of a course
To perform'd, you were my life was you see you from the grial
Then spur his manners play forewound again,
Where it butcher'st in heaven: he, after it,
Good bid for his sorrow lament lie on my name-blow;
Your noble tribunes prey, 
